# Parlay GRPO (after SFT) — Colab

Stage-2 GRPO uses [`training/grpo_train.py`](https://github.com/sh4shv4t/Parlay/blob/main/training/grpo_train.py): SFT-warmed LoRA + **train**-split prompts from your `jsonl`, with Parlay reward functions (ZOPA efficiency, ToM, anti-capitulation, format).

**Run order:** config → (Drive) → HF token (if needed) → Environment (pip + optional `jsonl` download) → clone → **GRPO train** → (optional push).

## SFT adapter
Set `SFT_DIR` to your local SFT output folder, or leave it wrong/missing and set `SFT_MODEL_HF` so the notebook downloads the adapter (e.g. `sh4shv4t/parlay-sft-1-5b`).

## JSONL
GRPO only loads rows with `"split": "train"`. If you see **0 remaining for GRPO**, add `split` in `generate_data` or check your file.

## T4 / VRAM
Default config uses `GRPO_G = 4` and `GRPO_STEPS = 100` for T4 demos; increase on A100 if you need longer runs. Lower `GRPO_G` further if you OOM.

## CUDA
Use **Runtime → GPU**. Avoid reinstalling `torch` on Colab unless necessary.


In [ ]:
# T4: use GRPO_G=4, GRPO_STEPS=100 (~2.5hrs). For A100 use G=8, steps=300.
# --- Set these, then run top to bottom ---
GITHUB_CLONE = "https://github.com/sh4shv4t/Parlay.git"
REPO_DIR = "/content/Parlay"

# Local SFT LoRA folder; if missing, we use SFT_MODEL_HF from the Hub
SFT_DIR = "/content/drive/MyDrive/ParlaySFT/sft_1.5b"
SFT_MODEL_HF = "sh4shv4t/parlay-sft-1-5b"

EPISODES_JSONL = "/content/episodes_v2.jsonl"
JSONL_IN_DRIVE = None  # e.g. "/content/drive/MyDrive/episodes_v2.jsonl"
JSONL_VIA_HF = None   # ("user/parlay-episodes", "episodes_v2.jsonl")

MIN_REWARD = -50.0
GRPO_STEPS = 100
GRPO_G = 4

OUTPUT_DIR = "/content/drive/MyDrive/ParlaySFT/grpo_1.5b"
HF_GRPO_REPO = "sh4shv4t/parlay-grpo-1-5b"
PUSH_TO_HF = True

from pathlib import Path as _P
if JSONL_IN_DRIVE:
    EPISODES_JSONL = str(_P(JSONL_IN_DRIVE))
print("Config OK.")


In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Not on Colab; skip Drive or mount manually.")


In [ ]:
import getpass
import os

def _get_hf_token() -> str:
    for k in ("HUGGINGFACE_HUB_TOKEN", "HF_TOKEN"):
        v = os.environ.get(k) or ""
        if v:
            return v
    try:
        from google.colab import userdata
        t = userdata.get("HF_TOKEN")
        if t:
            return t
    except Exception:
        pass
    return getpass.getpass("HuggingFace token (Hub download/push): ")

if PUSH_TO_HF or JSONL_VIA_HF:
    _t = _get_hf_token()
    if _t:
        os.environ["HF_TOKEN"] = _t
        os.environ["HUGGINGFACE_HUB_TOKEN"] = _t
        print("HF token set")
else:
    print("Optional: set HF token for private datasets.")


In [ ]:
import importlib.util
import subprocess
import sys

import torch
print("torch:", torch.__version__, "CUDA:", torch.cuda.is_available())

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers>=4.47.0", "trl>=0.14.0", "peft>=0.14.0", "accelerate>=1.2.0",
    "datasets>=3.2.0", "huggingface-hub>=0.24.0",
])

if JSONL_VIA_HF:
    from huggingface_hub import hf_hub_download
    _repo, _fn = JSONL_VIA_HF
    EPISODES_JSONL = hf_hub_download(repo_id=_repo, filename=_fn, repo_type="dataset")
    print("jsonl:", EPISODES_JSONL)
else:
    print("EPISODES_JSONL:", EPISODES_JSONL)


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

rd = Path(REPO_DIR)
if not (rd / "training" / "grpo_train.py").is_file():
    subprocess.check_call(["git", "clone", "--depth", "1", GITHUB_CLONE, str(rd)])
os.chdir(rd)
sys.path.insert(0, str(rd))
print("CWD:", os.getcwd())


In [ ]:
import json
import time
from pathlib import Path

from huggingface_hub import snapshot_download

data_path = Path(EPISODES_JSONL).resolve()
if not data_path.is_file():
    raise FileNotFoundError(f"Missing jsonl: {data_path}")

if SFT_DIR and Path(SFT_DIR).is_dir():
    model_in = str(Path(SFT_DIR).resolve())
    print("SFT local:", model_in)
else:
    print("Downloading SFT from Hub:", SFT_MODEL_HF)
    model_in = snapshot_download(SFT_MODEL_HF)

out = Path(OUTPUT_DIR).resolve()
out.mkdir(parents=True, exist_ok=True)

from training.grpo_train import train_grpo

t0 = time.time()
train_grpo(
    model_in,
    str(data_path),
    str(out),
    steps=int(GRPO_STEPS),
    min_reward=float(MIN_REWARD),
    num_generations=int(GRPO_G),
)
elapsed = time.time() - t0

summary = {
    "sft_source": model_in,
    "episodes_jsonl": str(data_path),
    "output_dir": str(out),
    "grpo_steps": GRPO_STEPS,
    "grpo_g": GRPO_G,
    "min_reward": MIN_REWARD,
    "train_seconds": round(elapsed, 1),
    "hf_repo": HF_GRPO_REPO if PUSH_TO_HF else None,
}
(out / "grpo_colab_run_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(json.dumps(summary, indent=2))


In [ ]:
import os
from pathlib import Path
if not PUSH_TO_HF:
    print("PUSH_TO_HF is False")
else:
    from huggingface_hub import HfApi
    tok = os.environ.get("HUGGINGFACE_HUB_TOKEN") or os.environ.get("HF_TOKEN")
    if not tok:
        raise ValueError("Set HF token for push")
    api = HfApi(token=tok)
    api.create_repo(repo_id=HF_GRPO_REPO, private=False, exist_ok=True, repo_type="model")
    api.upload_folder(
        folder_path=str(Path(OUTPUT_DIR).resolve()),
        repo_id=HF_GRPO_REPO,
        repo_type="model",
        commit_message="Parlay GRPO (Qwen2.5-1.5B) from Colab",
    )
    print("https://huggingface.co/" + HF_GRPO_REPO)


## If something breaks
- **Empty GRPO dataset:** only `"split": "train"` rows are used; fix JSONL or regenerate data.
- **TRL / `GRPOConfig` errors:** your `trl` version may differ; check `requirements-train.txt` and edit `grpo_train.py` args if needed.
- **OOM:** lower `GRPO_G` to 4, reduce `GRPO_STEPS`, or use a larger GPU.
